# Milestone 1 lab, notebook 5 of 8: The check card

## Where we are

| Notebook | What you do |
|---|---|
| 1. The question and five green checks | Meet Lotus Retails and the Monday question. Run the given pipeline for one night. Read its five checks |
| 2. Airflow runs it | Let Airflow run the same night. Read the DAG: its tasks and its arrows |
| 3. The number by hand | Add up one store's receipts yourself. Find out why the two numbers differ |
| 4. Who is hurt | For each table: who uses it, and what happens to them when it is wrong |
| **5. The check card** (you are here) | Meet the page you fill in before any check. Say what one row means |
| 6. Read the table first | Count the rows, find the smallest and largest values, count the empty cells |
| 7. Six questions for any table | Sort your worries under six questions. See which ones the five checks answer |
| 8. The production version | The same logic as a tested program |

**About ten minutes.**

In [1]:
from lab_helpers import show_doc, sql

## Step 1: The page that comes before a check

You now know two things about `sales_lines`: it holds a wrong row, and it is the most critical table Lotus has.

Before you write any check for it, you fill in one short page. It is called a **check card**. It has eight questions, and you will answer the same eight for every table in this project.

| # | Question |
|---|---|
| 1 | What does one row mean? |
| 2 | Who uses this table, and what do they decide with it? |
| 3 | How is the data made? So what can go wrong? |
| 4 | Which of those would hurt most? |
| 5 | What must be prepared before it can be checked? |
| 6 | The check: the rule, the limit, the lens, how serious |
| 7 | What happens when it fails? Who is told? |
| 8 | The proof: plant the fault, and show the check catching it |

Today you answer questions 1 to 5. The last three need the checks themselves, and those are milestone 2.

## Step 2: Question 1. What does one row mean?

Count three things in store `S03`'s Monday.

In [2]:
sql("""
    SELECT count(*)                   AS rows,
           count(DISTINCT receipt_id) AS receipts,
           count(DISTINCT line_id)    AS line_ids
    FROM clean.sales_lines
    WHERE store_id = 'S03' AND sales_date = '2026-08-24'
""")

,rows,receipts,line_ids
0,113,34,113


113 rows, and 34 receipts. So one row is **not** one receipt. One row is one item line on a receipt.

What one row means is called the **grain** of the table.

Here is why you say it out loud. Somebody asks: "How many customers did S03 serve on Monday?" A quick `count(*)` answers 113. The right answer is 34. Nothing fails. The number is three times too big.

Compare it with the grain of the output table.

In [3]:
sql("""
    SELECT * FROM output.daily_store_sales
    WHERE sales_date = '2026-08-24'
    ORDER BY store_id
""")

,sales_date,store_id,receipts,lines,net_sales_cents
0,2026-08-24,S01,40,117,79524
1,2026-08-24,S02,36,125,90128
2,2026-08-24,S03,34,113,79618
3,2026-08-24,S04,31,89,57977
4,2026-08-24,S05,22,68,47418
5,2026-08-24,S06,29,98,59519


There, one row is one store on one day. 113 rows of the first table became one row of the second.

The grain also tells you the **key**: the columns that name one row. The pipeline trusts `line_id`. But what names an item line in the real world is the receipt and its position on it.

In [4]:
sql("""
    SELECT count(*) AS rows,
           count(DISTINCT line_id) AS line_ids,
           count(DISTINCT receipt_id || ' / ' || CAST(line_no AS TEXT)) AS receipt_and_line_no
    FROM clean.sales_lines
""")

,rows,line_ids,receipt_and_line_no
0,20482,20482,20482


Both name every row, today. The card records which one is the true key, and why.

In [5]:
show_doc("sales_lines.md", "1. What does one row mean?")

**From `checks/sales_lines.md`, section 1. What does one row mean?**

One item line on one receipt. A receipt with three products has three rows.

The true key of a row is `receipt_id` + `line_no`. The column `line_id` is not a true key: the export program makes a new one every time it writes a file.

## Step 3: Question 2. Who uses it?

You answered this in notebook 4. On the card it becomes three rows.

In [6]:
show_doc("sales_lines.md", "2. Who uses this table, and what do they decide with it?")

**From `checks/sales_lines.md`, section 2. Who uses this table, and what do they decide with it?**

| Who | What they decide | What goes wrong for them when the data is bad |
|---|---|---|
| Store managers, through `daily_store_sales` | Nothing. They are **paid** from it: 1% of the week's net sales | Wrong pay. On 2026-08-24 store `S03` was $18.98 too high |
| Head of retail operations | She signs off the bonus every Monday | She signs a wrong number, and cannot say which one |
| The replenishment team, through `replenishment_list` | How much of each product to order for tomorrow | Too high wastes food. Too low leaves empty shelves |

Two outputs and three groups of people read this table. It is the most critical table Lotus has.

## Step 4: Questions 3 and 4. What can go wrong, and what hurts most?

Question 3 follows a sale from the cashier's hand to the file. Each step is made by a different person or machine, so each fails in its own way.

In [7]:
show_doc("sales_lines.md", "3. How is the data made? So what can go wrong?")

**From `checks/sales_lines.md`, section 3. How is the data made? So what can go wrong?**

| Step | Made by | What can go wrong |
|---|---|---|
| A cashier scans a product | A person and a till | A wrong quantity. A weight in the wrong unit |
| The till stamps the time | The till's clock | A wrong clock, so a wrong day |
| The till takes the price | The price list in the till | Yesterday's price |
| A customer brings something back | A person and a till | A refund with the wrong sign |
| The export program writes the night's file | A program in the store's back office | A line twice. A receipt twice. A whole file twice |
| The store sends the file | A back-office computer | No file. A late file |

That is a list of worries. Not every worry deserves a check. Question 4 ranks them by the harm to the people of question 2.

In [8]:
show_doc("sales_lines.md", "4. Which of those would hurt most?")

**From `checks/sales_lines.md`, section 4. Which of those would hurt most?**

Ranked by the money a person loses, and by how long it stays hidden.

| Rank | Worry | Why it is here |
|---|---|---|
| 1 | A store's whole day is missing or doubled | The largest error in pay: a full day of sales |
| 2 | Money with the wrong sign or the wrong size | One line can be larger than a day. Seen in notebook 3 |
| 3 | The same sale counted twice | Small each time, and it never shows as an error |
| 4 | A sale on the wrong day | It can fall out of the bonus week |
| 5 | A line the join drops | Sales are too low, and no row is left to look at |
| 6 | An empty `member_id` | It hurts nobody. A customer may pay without a loyalty card. **No check** |

Not every worry gets a check. Number 6 has no answer to question 2, so it gets none.

Look at the last row. An empty `member_id` hurts nobody: a customer may pay without a loyalty card. So it gets **no check**.

This is the rule of the card, and it is strict: **a check that has no answer to question 2 is deleted.** If you cannot name the person it protects, it only makes noise.

## Step 5: Question 5. What must be prepared first?

In [9]:
show_doc("sales_lines.md", "5. What must be prepared before it can be checked?")

**From `checks/sales_lines.md`, section 5. What must be prepared before it can be checked?**

| Worry | What must be prepared first |
|---|---|
| A whole day missing or doubled | Count the rows per store per day, and compare with a normal day for that store |
| Wrong sign or wrong size | Add the lines up per till per day, and hold the total against `till_totals` |
| The same sale twice | Group by `receipt_id` + `line_no`, not by `line_id` |
| A sale on the wrong day | Join to `stores` for the opening hours |
| A line the join drops | Compare the raw rows with the clean rows |

Four of the five need a join, a total or a comparison first. None of them is a one-line check.

Read the right-hand column. Count, add up, group, join, compare. Four of the five worries cannot be tested by one line on one column. The data must be prepared first.

That is why the five standard checks missed so much. Each of them is one line on one column.

## What you now know

- A check card is filled in **before** a check is written. It has eight questions.
- The **grain** is what one row of a table means. For `sales_lines` it is one item line on one receipt.
- Not every worry gets a check. A check with no reader to protect is deleted.
- Most useful checks need the data prepared first: a join, a total, or a comparison.

**Say it in an interview:** "Before I check a table I write down its grain, who reads it, how it is made, and which fault would hurt most. For the sales lines that ranking put a missing day at the top and an empty loyalty id at the bottom, with no check at all, because nobody is hurt by it."

**The theory:** [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), section "A starter kit for any table".

Next: [notebook 6](06_read_the_table_first.ipynb). The card lists what could go wrong. Now look at what is really in the table.